# Integração LoRa/Meshtastic, JSON e Alertas Ambientais

## Protótipo didático — trilha B: software-only / simulação

Este notebook demonstra uma cadeia local de dados ambientais: geração de leituras sintéticas, validação de JSON, cálculo de tendência, classificação de risco, mensagem curta de alerta e preparação de um envelope de integração simulado.

> **Limite importante:** não há hardware LoRa, sensores, GPS, nós Meshtastic, aplicativo móvel, broker MQTT ou transmissão de rádio nesta execução. Todos os dados usam `source: "simulation"`; as saídas validam a lógica de software, não uma comunicação física.

## Referências e distinções conceituais

- **LoRa** é a tecnologia de rádio de longo alcance e baixo consumo.
- **LoRaWAN** é uma arquitetura/protocolo LPWAN, normalmente em estrela, com nós, gateways e servidor de rede. O artigo principal usa essa arquitetura.
- **Meshtastic** usa LoRa em uma malha descentralizada; não é LoRaWAN e não deve ser descrito como tal.
- **MQTT/JSON** pode ser uma ponte IP entre borda e sistemas de dados; não substitui a comunicação LoRa nem demonstra transmissão pelo ar.

Fontes de estudo: [atividade do Período 8](../documentos_referencia/), [artigo LoRaWAN de 2023](https://doi.org/10.1016/j.iotcps.2023.04.005), [artigo Meshtastic de 2026](https://arxiv.org/abs/2605.20379), [MQTT no Meshtastic](https://meshtastic.org/docs/configuration/module/mqtt/) e [telemetria Meshtastic](https://meshtastic.org/docs/configuration/module/telemetry/).

A opção JSON do MQTT facilita integrações externas, mas a documentação do Meshtastic alerta que esses pacotes não são criptografados. Por isso, este exemplo não usa rede, credenciais, localização real ou broker público.

## Arquitetura da simulação

```text
[leituras sintéticas] -> [JSON canônico] -> [validador] -> [motor de risco]
                                                        |
                                                        v
[evidências locais] <- [exportador] <- [envelope MQTT simulado] <- [alerta curto]

Fronteira da simulação: nenhum pacote é codificado para rádio, enviado por LoRa,
publicado em MQTT ou exibido por um aplicativo Meshtastic.
```

A estrutura segue as quatro camadas discutidas na referência complementar: percepção (leitura), rede (contrato de mensagem), borda/processamento (validação e regra) e aplicação (alerta e evidência).

In [1]:
# Pré-requisitos: execute o notebook a partir da raiz do repositório.
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((candidate for candidate in (current_directory, *current_directory.parents)
                     if (candidate / 'notebooks').is_dir() and (candidate / 'planejamento.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Não foi possível localizar a raiz do repositório a partir deste diretório.')

print(f'Python: {sys.version.split()[0]}')
print(f'Raiz do projeto: {PROJECT_ROOT}')
print('Modo de dados obrigatório: simulation')

Python: 3.14.4
Raiz do projeto: /home/mateus-tatsch/Documents/Americas TechGuard/Semana08/lora_meshtastic_integration_json_environmental_alerts_mobile_devices
Modo de dados obrigatório: simulation


In [2]:
# O pipeline usa somente a biblioteca padrão do Python.
from collections import Counter
from datetime import datetime, timedelta
from typing import Any, Mapping
import json

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
print('Imports e caminhos configurados.')

Imports e caminhos configurados.


In [3]:
# Configuração explícita e versionada da PoC.
SCHEMA_VERSION = '1.0'
SIMULATION_SOURCE = 'simulation'
SENSOR_TYPE = 'rainfall'
SENSOR_UNIT = 'mm'
MAX_ALERT_LENGTH = 140
RAIN_GAUGE_TIP_MM = 0.2
SAMPLE_INTERVAL_MINUTES = 5
ACCUMULATION_WINDOW_MINUTES = 60
PAYLOAD_BUDGET_BYTES = 250
RISK_THRESHOLDS_MM_1H = {
    'safe_upper': 5,
    'attention_upper': 20,
    'alert_upper': 40,
}

print('Faixas didáticas de chuva acumulada em 1 h:', RISK_THRESHOLDS_MM_1H)
print(f'Pluviômetro de caçamba basculante: {RAIN_GAUGE_TIP_MM:.1f} mm por basculamento.')
print(f'Orçamento didático de transporte: {PAYLOAD_BUDGET_BYTES} bytes.')
print('Observação: os limiares não são calibração operacional de defesa civil.')

Faixas didáticas de chuva acumulada em 1 h: {'safe_upper': 5, 'attention_upper': 20, 'alert_upper': 40}
Pluviômetro de caçamba basculante: 0.2 mm por basculamento.
Orçamento didático de transporte: 250 bytes.
Observação: os limiares não são calibração operacional de defesa civil.


## Contrato JSON

Cada entrada possui `schema_version`, `reading_id`, `device_id`, `timestamp`, `latitude`, `longitude`, `sensor_type`, `sensor_value`, `unit` e `source`. A simulação também guarda `sensor_simulation` (basculamentos, chuva no intervalo e estado). O processamento adiciona `rainfall_intensity_mm_per_h`, `rainfall_accumulated_mm_1h`, `risk_level`, `alert_message` e `processing_status`.

O contrato preserva os campos mínimos exigidos pela atividade. `risk_level` e `alert_message` são derivados pelo código, não recebidos como verdade de uma fonte externa. As coordenadas abaixo são sintéticas e servem somente para demonstrar formato e validação.

In [4]:
def simulate_rain_gauge_measurement(
    true_rainfall_mm: float, *, fault: str | None = None
) -> dict[str, Any]:
    """Simula caçamba basculante: chuva no intervalo -> contagem inteira -> mm."""
    if fault == 'timeout':
        return {
            'status': 'timeout', 'tip_count': None, 'rainfall_mm': None,
            'reason': 'Nenhuma leitura recebida no intervalo esperado.',
        }
    if fault == 'stuck':
        return {
            'status': 'stuck', 'tip_count': 0, 'rainfall_mm': 0.0,
            'reason': 'Contador travado apesar de chuva sintética.',
        }
    if fault == 'counter_reset':
        return {
            'status': 'counter_reset', 'tip_count': 0, 'rainfall_mm': None,
            'reason': 'Reset de contador injetado para teste.',
        }

    tip_count = round(true_rainfall_mm / RAIN_GAUGE_TIP_MM)
    rainfall_mm = round(tip_count * RAIN_GAUGE_TIP_MM, 1)
    return {
        'status': 'ok',
        'tip_count': tip_count,
        'rainfall_mm': rainfall_mm,
        'reason': None,
    }

def make_simulated_payload(reading_id: str, timestamp: str, measurement: Mapping[str, Any]) -> dict[str, Any]:
    return {
        'schema_version': SCHEMA_VERSION,
        'reading_id': reading_id,
        'device_id': 'sim-rain-node-01',
        'node_name': 'Pluviômetro simulado 01',
        'timestamp': timestamp,
        'latitude': -26.3000,
        'longitude': -48.8400,
        'altitude_m': None,
        'sensor_type': SENSOR_TYPE,
        'sensor_value': measurement['rainfall_mm'],
        'unit': SENSOR_UNIT,
        'source': SIMULATION_SOURCE,
        'sensor_simulation': dict(measurement),
    }

def build_synthetic_readings() -> list[dict[str, Any]]:
    """Cria cenários determinísticos de chuva medida por basculamentos."""
    observations = [
        ('sim-20260716-0001', '2026-07-16T12:00:00-03:00', 0.0),
        ('sim-20260716-0002', '2026-07-16T12:05:00-03:00', 5.0),
        ('sim-20260716-0003', '2026-07-16T12:10:00-03:00', 16.0),
        ('sim-20260716-0004', '2026-07-16T12:15:00-03:00', 24.0),
    ]
    return [
        make_simulated_payload(reading_id, timestamp, simulate_rain_gauge_measurement(rainfall_mm))
        for reading_id, timestamp, rainfall_mm in observations
    ]

synthetic_readings = build_synthetic_readings()
sensor_fault_payloads = [
    make_simulated_payload('sensor-timeout', '2026-07-16T12:20:00-03:00', simulate_rain_gauge_measurement(12.0, fault='timeout')),
    make_simulated_payload('sensor-stuck', '2026-07-16T12:25:00-03:00', simulate_rain_gauge_measurement(12.0, fault='stuck')),
    make_simulated_payload('sensor-counter-reset', '2026-07-16T12:30:00-03:00', simulate_rain_gauge_measurement(12.0, fault='counter_reset')),
]
print('Leituras válidas do pluviômetro simulado:')
for reading in synthetic_readings:
    sensor = reading['sensor_simulation']
    print(f"- {reading['reading_id']}: basculamentos={sensor['tip_count']}, chuva={reading['sensor_value']} mm")
print('Falhas de sensor injetadas:', [item['sensor_simulation']['status'] for item in sensor_fault_payloads])

Leituras válidas do pluviômetro simulado:
- sim-20260716-0001: basculamentos=0, chuva=0.0 mm
- sim-20260716-0002: basculamentos=25, chuva=5.0 mm
- sim-20260716-0003: basculamentos=80, chuva=16.0 mm
- sim-20260716-0004: basculamentos=120, chuva=24.0 mm
Falhas de sensor injetadas: ['timeout', 'stuck', 'counter_reset']


In [5]:
REQUIRED_INPUT_FIELDS = {
    'schema_version', 'reading_id', 'device_id', 'timestamp', 'latitude', 'longitude',
    'sensor_type', 'sensor_value', 'unit', 'source'
}

def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)

def validate_payload(payload: Mapping[str, Any]) -> list[dict[str, str]]:
    """Valida dados de entrada e devolve erros estruturados, sem lançar para o lote."""
    if not isinstance(payload, Mapping):
        return [{'field': 'payload', 'message': 'O payload precisa ser um objeto JSON.'}]

    errors: list[dict[str, str]] = []
    for field in sorted(REQUIRED_INPUT_FIELDS - set(payload)):
        errors.append({'field': field, 'message': 'Campo obrigatório ausente.'})

    if errors:
        return errors

    if payload['schema_version'] != SCHEMA_VERSION:
        errors.append({'field': 'schema_version', 'message': f'Versão esperada: {SCHEMA_VERSION}.'})
    for field in ('reading_id', 'device_id'):
        if not isinstance(payload[field], str) or not payload[field].strip():
            errors.append({'field': field, 'message': 'Deve ser texto não vazio.'})

    try:
        parsed_timestamp = datetime.fromisoformat(str(payload['timestamp']))
        if parsed_timestamp.tzinfo is None:
            errors.append({'field': 'timestamp', 'message': 'Use ISO 8601 com fuso horário.'})
    except ValueError:
        errors.append({'field': 'timestamp', 'message': 'Timestamp ISO 8601 inválido.'})

    latitude, longitude = payload['latitude'], payload['longitude']
    if not is_number(latitude) or not -90 <= latitude <= 90:
        errors.append({'field': 'latitude', 'message': 'Latitude deve estar entre -90 e 90.'})
    if not is_number(longitude) or not -180 <= longitude <= 180:
        errors.append({'field': 'longitude', 'message': 'Longitude deve estar entre -180 e 180.'})
    if payload['sensor_type'] != SENSOR_TYPE:
        errors.append({'field': 'sensor_type', 'message': f'Tipo aceito nesta PoC: {SENSOR_TYPE}.'})
    if not is_number(payload['sensor_value']) or payload['sensor_value'] < 0:
        errors.append({'field': 'sensor_value', 'message': 'Valor deve ser numérico e não negativo.'})
    if payload['unit'] != SENSOR_UNIT:
        errors.append({'field': 'unit', 'message': f'Unidade esperada: {SENSOR_UNIT}.'})
    if payload['source'] != SIMULATION_SOURCE:
        errors.append({'field': 'source', 'message': 'Esta execução aceita somente dados de simulation.'})
    sensor_metadata = payload.get('sensor_simulation')
    if isinstance(sensor_metadata, Mapping) and sensor_metadata.get('status') != 'ok':
        errors.append({'field': 'sensor_simulation.status', 'message': f"Falha de sensor simulada: {sensor_metadata.get('status')}."})
    return errors

valid_errors = validate_payload(synthetic_readings[0])
assert valid_errors == []
print('Payload válido: nenhuma inconsistência encontrada.')

Payload válido: nenhuma inconsistência encontrada.


In [6]:
# Falhas de formato e de sensor são processadas didaticamente; não interrompem o lote principal.
invalid_payloads = [
    {**synthetic_readings[0], 'reading_id': 'invalid-timestamp', 'timestamp': '16/07/2026 12:00'},
    {**synthetic_readings[0], 'reading_id': 'invalid-location', 'latitude': -91.0},
    {**synthetic_readings[0], 'reading_id': 'invalid-unit', 'unit': 'meters'},
]

all_invalid_payloads = [*invalid_payloads, *sensor_fault_payloads]
for payload in all_invalid_payloads:
    print(f"{payload['reading_id']}: {validate_payload(payload)}")

invalid-timestamp: [{'field': 'timestamp', 'message': 'Timestamp ISO 8601 inválido.'}]
invalid-location: [{'field': 'latitude', 'message': 'Latitude deve estar entre -90 e 90.'}]
invalid-unit: [{'field': 'unit', 'message': 'Unidade esperada: mm.'}]
sensor-timeout: [{'field': 'sensor_value', 'message': 'Valor deve ser numérico e não negativo.'}, {'field': 'sensor_simulation.status', 'message': 'Falha de sensor simulada: timeout.'}]
sensor-stuck: [{'field': 'sensor_simulation.status', 'message': 'Falha de sensor simulada: stuck.'}]
sensor-counter-reset: [{'field': 'sensor_value', 'message': 'Valor deve ser numérico e não negativo.'}, {'field': 'sensor_simulation.status', 'message': 'Falha de sensor simulada: counter_reset.'}]


In [7]:
def parse_timestamp(payload: Mapping[str, Any]) -> datetime:
    return datetime.fromisoformat(str(payload['timestamp']))

def calculate_rainfall_intensity(current: Mapping[str, Any]) -> float:
    """Converte a chuva do intervalo de amostragem para intensidade em mm/h."""
    return round(float(current['sensor_value']) * 60 / SAMPLE_INTERVAL_MINUTES, 1)

def calculate_accumulated_rainfall(
    accepted_history: list[Mapping[str, Any]], current: Mapping[str, Any]
) -> float:
    """Soma leituras aceitas na janela móvel de uma hora, incluindo a atual."""
    current_time = parse_timestamp(current)
    window_start = current_time - timedelta(minutes=ACCUMULATION_WINDOW_MINUTES)
    relevant = [
        reading for reading in accepted_history
        if parse_timestamp(reading) >= window_start
    ]
    return round(sum(float(reading['sensor_value']) for reading in relevant) + float(current['sensor_value']), 1)

for reading in synthetic_readings:
    print(reading['reading_id'], '->', calculate_rainfall_intensity(reading), 'mm/h (intensidade do intervalo)')

sim-20260716-0001 -> 0.0 mm/h (intensidade do intervalo)
sim-20260716-0002 -> 60.0 mm/h (intensidade do intervalo)
sim-20260716-0003 -> 192.0 mm/h (intensidade do intervalo)
sim-20260716-0004 -> 288.0 mm/h (intensidade do intervalo)


In [8]:
def classify_risk(rainfall_accumulated_mm_1h: float) -> str:
    """Aplica política didática baseada na chuva acumulada em uma hora."""
    if rainfall_accumulated_mm_1h < RISK_THRESHOLDS_MM_1H['safe_upper']:
        return 'safe'
    if rainfall_accumulated_mm_1h < RISK_THRESHOLDS_MM_1H['attention_upper']:
        return 'attention'
    if rainfall_accumulated_mm_1h < RISK_THRESHOLDS_MM_1H['alert_upper']:
        return 'alert'
    return 'critical'

boundary_cases = {4.9: 'safe', 5: 'attention', 19.9: 'attention', 20: 'alert', 39.9: 'alert', 40: 'critical'}
for accumulated_rainfall, expected_risk in boundary_cases.items():
    actual_risk = classify_risk(accumulated_rainfall)
    assert actual_risk == expected_risk
    print(f'{accumulated_rainfall:>4} mm/1h -> {actual_risk}')

print('Bordas de risco validadas.')

 4.9 mm/1h -> safe
   5 mm/1h -> attention
19.9 mm/1h -> attention
  20 mm/1h -> alert
39.9 mm/1h -> alert
  40 mm/1h -> critical
Bordas de risco validadas.


In [9]:
def build_alert_message(rainfall_accumulated_mm_1h: float, risk_level: str) -> str:
    """Gera texto curto; não substitui orientações oficiais de emergência."""
    accumulated = f'{rainfall_accumulated_mm_1h:.1f}'
    messages = {
        'safe': f'Chuva acumulada em 1 h: {accumulated} mm. Monitoramento simulado ativo.',
        'attention': f'ATENÇÃO: chuva acumulada em 1 h: {accumulated} mm. Acompanhe os avisos oficiais.',
        'alert': f'ALERTA: chuva acumulada em 1 h: {accumulated} mm. Acompanhe os avisos oficiais.',
        'critical': f'CRÍTICO: chuva acumulada em 1 h: {accumulated} mm. Siga os avisos oficiais.',
    }
    message = messages[risk_level]
    if len(message) > MAX_ALERT_LENGTH:
        raise ValueError('Mensagem excede o limite didático configurado.')
    return message

for risk in ('safe', 'attention', 'alert', 'critical'):
    print(build_alert_message(45.0, risk))

Chuva acumulada em 1 h: 45.0 mm. Monitoramento simulado ativo.
ATENÇÃO: chuva acumulada em 1 h: 45.0 mm. Acompanhe os avisos oficiais.
ALERTA: chuva acumulada em 1 h: 45.0 mm. Acompanhe os avisos oficiais.
CRÍTICO: chuva acumulada em 1 h: 45.0 mm. Siga os avisos oficiais.


In [10]:
def rejected_result(payload: Mapping[str, Any], errors: list[dict[str, str]]) -> dict[str, Any]:
    return {
        **dict(payload),
        'processing_status': 'rejected',
        'errors': errors,
    }

def process_reading(
    payload: Mapping[str, Any],
    accepted_history: list[Mapping[str, Any]] | None = None,
    seen_ids: set[str] | None = None,
) -> tuple[dict[str, Any], list[Mapping[str, Any]]]:
    """Valida, calcula chuva acumulada e devolve o histórico de leituras aceitas."""
    accepted_history = list(accepted_history or [])
    errors = validate_payload(payload)
    if errors:
        return rejected_result(payload, errors), accepted_history
    if seen_ids is not None and payload['reading_id'] in seen_ids:
        return rejected_result(payload, [{'field': 'reading_id', 'message': 'Leitura duplicada no lote.'}]), accepted_history

    try:
        if accepted_history and parse_timestamp(payload) <= parse_timestamp(accepted_history[-1]):
            raise ValueError('O timestamp atual deve ser posterior ao da última leitura aceita.')
        intensity = calculate_rainfall_intensity(payload)
        accumulated = calculate_accumulated_rainfall(accepted_history, payload)
    except ValueError as error:
        return rejected_result(payload, [{'field': 'timestamp', 'message': str(error)}]), accepted_history

    risk_level = classify_risk(accumulated)
    result = {
        **dict(payload),
        'rainfall_intensity_mm_per_h': intensity,
        'rainfall_accumulated_mm_1h': accumulated,
        'risk_level': risk_level,
        'alert_message': build_alert_message(accumulated, risk_level),
        'processing_status': 'accepted',
    }
    return result, [*accepted_history, result]

accepted_example, _ = process_reading(synthetic_readings[0], [])
rejected_example, _ = process_reading(invalid_payloads[0])
print('Aceito:', accepted_example['risk_level'], '| Rejeitado:', rejected_example['errors'][0]['field'])

Aceito: safe | Rejeitado: timestamp


In [11]:
def process_batch(payloads: list[Mapping[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    results: list[dict[str, Any]] = []
    accepted_history: list[Mapping[str, Any]] = []
    seen_ids: set[str] = set()

    for payload in payloads:
        result, accepted_history = process_reading(payload, accepted_history, seen_ids)
        results.append(result)
        if result['processing_status'] == 'accepted':
            seen_ids.add(result['reading_id'])

    accepted = [item for item in results if item['processing_status'] == 'accepted']
    rejected = [item for item in results if item['processing_status'] == 'rejected']
    summary = {
        'simulation': True,
        'input_count': len(payloads),
        'accepted_count': len(accepted),
        'rejected_count': len(rejected),
        'risk_counts': dict(Counter(item['risk_level'] for item in accepted)),
    }
    return results, summary

processed_results, batch_summary = process_batch(synthetic_readings)
assert batch_summary['accepted_count'] == 4
assert batch_summary['risk_counts'] == {'safe': 1, 'attention': 1, 'alert': 1, 'critical': 1}
print(json.dumps(batch_summary, ensure_ascii=False, indent=2))

{
  "simulation": true,
  "input_count": 4,
  "accepted_count": 4,
  "rejected_count": 0,
  "risk_counts": {
    "safe": 1,
    "attention": 1,
    "alert": 1,
    "critical": 1
  }
}


In [12]:
# Injeção determinística de falhas de transmissão; não modela qualidade de rádio real.
TRANSMISSION_PLAN = {
    'sim-20260716-0001': [('delivered', 0.4)],
    'sim-20260716-0002': [('lost', None)],
    'sim-20260716-0003': [('delayed', 4.5)],
    'sim-20260716-0004': [('delivered', 0.8), ('duplicate', 1.1)],
}

def simulate_transmission_events(payloads: list[Mapping[str, Any]]) -> list[dict[str, Any]]:
    events: list[dict[str, Any]] = []
    for payload in payloads:
        for attempt, (outcome, delay_s) in enumerate(TRANSMISSION_PLAN[payload['reading_id']], start=1):
            events.append({
                'simulation': True,
                'reading_id': payload['reading_id'],
                'attempt': attempt,
                'planned_outcome': outcome,
                'planned_delay_s': delay_s,
            })
    return events

def simulate_gateway_receipts(events: list[Mapping[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, int]]:
    seen_readings: set[str] = set()
    receipts: list[dict[str, Any]] = []
    for event in events:
        if event['planned_outcome'] == 'lost':
            disposition = 'not_received'
        elif event['reading_id'] in seen_readings:
            disposition = 'ignored_duplicate'
        else:
            seen_readings.add(event['reading_id'])
            disposition = 'received'
        receipts.append({**dict(event), 'gateway_disposition': disposition})

    counts = Counter(item['gateway_disposition'] for item in receipts)
    summary = {
        'simulation': True,
        'planned_events': len(receipts),
        'received_events': counts['received'],
        'not_received_events': counts['not_received'],
        'ignored_duplicate_events': counts['ignored_duplicate'],
    }
    return receipts, summary

transmission_events = simulate_transmission_events(processed_results)
transmission_receipts, transmission_summary = simulate_gateway_receipts(transmission_events)
assert transmission_summary == {
    'simulation': True, 'planned_events': 5, 'received_events': 3,
    'not_received_events': 1, 'ignored_duplicate_events': 1,
}
print('Resumo de falhas de transmissão simuladas:', transmission_summary)

def serialized_size_bytes(content: Mapping[str, Any]) -> int:
    return len(json.dumps(content, ensure_ascii=False, separators=(',', ':')).encode('utf-8'))

def compact_for_transport(payload: Mapping[str, Any]) -> dict[str, Any]:
    """Contrato compacto ilustrativo; não é o formato binário do Meshtastic."""
    return {
        'i': payload['reading_id'],
        't': payload['timestamp'],
        'p': [payload['latitude'], payload['longitude']],
        's': 'rg',
        'v': payload['sensor_value'],
        'u': payload['unit'],
        'a': payload['rainfall_accumulated_mm_1h'],
        'r': payload['risk_level'],
        'm': payload['alert_message'],
        'o': 'sim',
    }

payload_size_audit = []
for payload in processed_results:
    compact_payload = compact_for_transport(payload)
    canonical_bytes = serialized_size_bytes(payload)
    compact_bytes = serialized_size_bytes(compact_payload)
    payload_size_audit.append({
        'reading_id': payload['reading_id'],
        'budget_bytes': PAYLOAD_BUDGET_BYTES,
        'canonical_json_bytes': canonical_bytes,
        'compact_json_bytes': compact_bytes,
        'canonical_within_budget': canonical_bytes <= PAYLOAD_BUDGET_BYTES,
        'compact_within_budget': compact_bytes <= PAYLOAD_BUDGET_BYTES,
        'compact_payload': compact_payload,
    })

assert all(item['compact_within_budget'] for item in payload_size_audit)
assert all(not item['canonical_within_budget'] for item in payload_size_audit)
print('Auditoria de tamanho (bytes UTF-8, orçamento apenas didático):')
for item in payload_size_audit:
    print(f"- {item['reading_id']}: canônico={item['canonical_json_bytes']} | compacto={item['compact_json_bytes']}")

def to_meshtastic_mqtt_simulation(processed_payload: Mapping[str, Any]) -> dict[str, Any]:
    """Cria contrato ilustrativo de borda; não abre conexão e não codifica pacote de rádio."""
    if processed_payload['processing_status'] != 'accepted':
        raise ValueError('Somente leituras aceitas podem ser preparadas para integração simulada.')
    return {
        'simulation': True,
        'transport': 'MQTT JSON bridge illustration — not a Meshtastic radio packet',
        'topic': f'americas-techguard/simulation/{processed_payload["device_id"]}/telemetry',
        'payload': dict(processed_payload),
        'warning': 'Nenhuma publicação MQTT, transmissão LoRa ou notificação móvel ocorreu.',
    }

alert_example = next(item for item in processed_results if item['risk_level'] == 'alert')
simulated_envelope = to_meshtastic_mqtt_simulation(alert_example)
print(json.dumps(simulated_envelope, ensure_ascii=False, indent=2))

Resumo de falhas de transmissão simuladas: {'simulation': True, 'planned_events': 5, 'received_events': 3, 'not_received_events': 1, 'ignored_duplicate_events': 1}
Auditoria de tamanho (bytes UTF-8, orçamento apenas didático):
- sim-20260716-0001: canônico=579 | compacto=199
- sim-20260716-0002: canônico=597 | compacto=215
- sim-20260716-0003: canônico=595 | compacto=211
- sim-20260716-0004: canônico=596 | compacto=211
{
  "simulation": true,
  "transport": "MQTT JSON bridge illustration — not a Meshtastic radio packet",
  "topic": "americas-techguard/simulation/sim-rain-node-01/telemetry",
  "payload": {
    "schema_version": "1.0",
    "reading_id": "sim-20260716-0003",
    "device_id": "sim-rain-node-01",
    "node_name": "Pluviômetro simulado 01",
    "timestamp": "2026-07-16T12:10:00-03:00",
    "latitude": -26.3,
    "longitude": -48.84,
    "altitude_m": null,
    "sensor_type": "rainfall",
    "sensor_value": 16.0,
    "unit": "mm",
    "source": "simulation",
    "sensor_simul

## Limite do adaptador simulado

O envelope anterior representa uma possível mensagem entregue a uma camada de borda depois de uma integração real. Ele **não** é um pacote de rádio Meshtastic, não representa protobuf, não calcula Time-on-Air e não executa *managed flooding*. As perdas, atrasos e duplicatas anteriores são falhas injetadas por cenário, não métricas de RSSI, SNR, PDR ou latência medidas.

A auditoria de tamanho mede somente bytes UTF-8 do JSON e compara um contrato canônico a uma versão compacta sob orçamento didático. Ela não mede limite de payload, fragmentação ou tempo no ar de um rádio real. Em uma futura etapa física, o adaptador deverá ser substituído por integração documentada com dispositivos reais, canal configurado, firmware compatível e broker protegido.

In [13]:
def print_results_table(rows: list[Mapping[str, Any]]) -> None:
    columns = [
        ('reading_id', 'Leitura'),
        ('sensor_value', 'Chuva (mm)'),
        ('rainfall_intensity_mm_per_h', 'Intensidade'),
        ('rainfall_accumulated_mm_1h', 'Acum. 1h'),
        ('risk_level', 'Risco'),
        ('processing_status', 'Estado'),
    ]
    rendered = []
    for row in rows:
        rendered.append([str(row.get(key, '')) for key, _ in columns])
    widths = [max(len(label), *(len(row[index]) for row in rendered)) for index, (_, label) in enumerate(columns)]
    header = ' | '.join(label.ljust(widths[index]) for index, (_, label) in enumerate(columns))
    separator = '-+-'.join('-' * width for width in widths)
    print(header)
    print(separator)
    for row in rendered:
        print(' | '.join(value.ljust(widths[index]) for index, value in enumerate(row)))

print_results_table(processed_results)
print('\nMensagens de alerta:')
for result in processed_results:
    print(f"- [{result['risk_level']}] {result['alert_message']}")

print('\nRecepção de falhas de transmissão simuladas:')
for receipt in transmission_receipts:
    print(f"- {receipt['reading_id']} tentativa {receipt['attempt']}: {receipt['planned_outcome']} -> {receipt['gateway_disposition']}")

print('\nAuditoria de payload (bytes UTF-8):')
for item in payload_size_audit:
    print(f"- {item['reading_id']}: {item['canonical_json_bytes']} bytes canônico | {item['compact_json_bytes']} bytes compacto")

Leitura           | Chuva (mm) | Intensidade | Acum. 1h | Risco     | Estado  
------------------+------------+-------------+----------+-----------+---------
sim-20260716-0001 | 0.0        | 0.0         | 0.0      | safe      | accepted
sim-20260716-0002 | 5.0        | 60.0        | 5.0      | attention | accepted
sim-20260716-0003 | 16.0       | 192.0       | 21.0     | alert     | accepted
sim-20260716-0004 | 24.0       | 288.0       | 45.0     | critical  | accepted

Mensagens de alerta:
- [safe] Chuva acumulada em 1 h: 0.0 mm. Monitoramento simulado ativo.
- [attention] ATENÇÃO: chuva acumulada em 1 h: 5.0 mm. Acompanhe os avisos oficiais.
- [alert] ALERTA: chuva acumulada em 1 h: 21.0 mm. Acompanhe os avisos oficiais.
- [critical] CRÍTICO: chuva acumulada em 1 h: 45.0 mm. Siga os avisos oficiais.

Recepção de falhas de transmissão simuladas:
- sim-20260716-0001 tentativa 1: delivered -> received
- sim-20260716-0002 tentativa 1: lost -> not_received
- sim-20260716-0003 tentativa 1:

In [14]:
def write_json(path: Path, content: Any) -> Path:
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return path

OUTPUT_DIR.mkdir(exist_ok=True)
rejected_results = [process_reading(payload)[0] for payload in all_invalid_payloads]
sensor_fault_results = [item for item in rejected_results if item['reading_id'].startswith('sensor-')]
output_files = {
    'processed': write_json(OUTPUT_DIR / 'processed_readings.json', processed_results),
    'rejected': write_json(OUTPUT_DIR / 'rejected_readings.json', rejected_results),
    'summary': write_json(OUTPUT_DIR / 'execution_summary.json', batch_summary),
    'sensor_faults': write_json(OUTPUT_DIR / 'sensor_faults_simulated.json', sensor_fault_results),
    'transmission': write_json(OUTPUT_DIR / 'transmission_faults_simulated.json', {
        'summary': transmission_summary, 'events': transmission_events, 'receipts': transmission_receipts,
    }),
    'payload_audit': write_json(OUTPUT_DIR / 'payload_size_audit.json', payload_size_audit),
    'envelope': write_json(OUTPUT_DIR / 'meshtastic_mqtt_envelope_simulated.json', simulated_envelope),
}
for label, path in output_files.items():
    print(f'{label}: {path.relative_to(PROJECT_ROOT)}')

processed: outputs/processed_readings.json
rejected: outputs/rejected_readings.json
summary: outputs/execution_summary.json
sensor_faults: outputs/sensor_faults_simulated.json
transmission: outputs/transmission_faults_simulated.json
payload_audit: outputs/payload_size_audit.json
envelope: outputs/meshtastic_mqtt_envelope_simulated.json


In [15]:
# Verificação de persistência: relê evidências gravadas pelo notebook.
loaded_processed = json.loads(output_files['processed'].read_text(encoding='utf-8'))
loaded_rejected = json.loads(output_files['rejected'].read_text(encoding='utf-8'))
loaded_summary = json.loads(output_files['summary'].read_text(encoding='utf-8'))
loaded_sensor_faults = json.loads(output_files['sensor_faults'].read_text(encoding='utf-8'))
loaded_transmission = json.loads(output_files['transmission'].read_text(encoding='utf-8'))
loaded_payload_audit = json.loads(output_files['payload_audit'].read_text(encoding='utf-8'))

assert len(loaded_processed) == loaded_summary['accepted_count']
assert len(loaded_rejected) == len(all_invalid_payloads)
assert len(loaded_sensor_faults) == len(sensor_fault_payloads)
assert loaded_transmission['summary'] == transmission_summary
assert loaded_transmission['summary']['ignored_duplicate_events'] == 1
assert all(item['compact_within_budget'] for item in loaded_payload_audit)
assert {item['risk_level'] for item in loaded_processed} == {'safe', 'attention', 'alert', 'critical'}
assert all(item['source'] == SIMULATION_SOURCE for item in loaded_processed)
print('Evidências relidas com sucesso.')
print('Contagens:', loaded_summary)

Evidências relidas com sucesso.
Contagens: {'simulation': True, 'input_count': 4, 'accepted_count': 4, 'rejected_count': 0, 'risk_counts': {'safe': 1, 'attention': 1, 'alert': 1, 'critical': 1}}


## Mapeamento para uma etapa física futura

Com acesso autorizado em Joinville, o contrato acima pode ser reutilizado com uma T-Beam V1.1, Heltec LoRa 32 V3 ou TTGO LoRa T3S3 1.2 configurada para 915 MHz. A sequência mínima seria: confirmar a revisão do rádio e compatibilidade do firmware; usar pelo menos dois nós (`CLIENT` e `ROUTER`/`GATEWAY`); instalar antena adequada antes de transmitir; validar serial e BLE; calibrar o sensor; e somente então medir RSSI, SNR, PDR e atraso.

A revisão do hardware é essencial: variantes de placas podem usar transceptores diferentes. O sensor pluviométrico também não deve ser assumido como telemetria nativa sem conferir suporte do firmware. Em uma integração MQTT real, utilizar broker próprio, TLS e credenciais fora do Git.

## Resultado, limitações e próximos passos

Esta PoC valida que leituras sintéticas podem ser derivadas de uma simulação de pluviômetro de caçamba basculante, estruturadas em JSON, rejeitadas quando inválidas ou quando o sensor sinaliza falha, enriquecidas com intensidade, chuva acumulada em 1 h e risco, transformadas em mensagem curta e exportadas como evidência. Ela também demonstra perda, atraso e duplicata por injeção de cenário e compara o tamanho UTF-8 de payloads canônicos e compactos. Apoia o Americas TechGuard como uma camada intermediária entre futura instrumentação ambiental, processamento, dashboards e disseminação de alertas.

Não foram validados: precisão do sensor, GPS, autonomia, rádio, alcance, malha, interferência, MQTT, segurança de transporte, aplicativo móvel, entendimento do alerta por usuários ou limiares operacionais. As falhas de transmissão e o orçamento de payload são apenas modelos de software, não medições físicas. Os próximos passos são testar os mesmos contratos com hardware autorizado, medir o enlace em campo e definir regras de risco com dados e responsáveis competentes.